In [ ]:
import pyedflib
import numpy as np
import gridfs
from pymongo import MongoClient
from datetime import datetime
import io
import xml.etree.ElementTree as ET
import os
import glob

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches


# GLOBAL CACHE - Tüm program boyunca geçerli
_STAGES_CACHE = {}


def clear_stages_cache():
    """
    Cache'i temizle (ihtiyaç olursa)
    """
    global _STAGES_CACHE
    _STAGES_CACHE.clear()
    print("✓ Stage cache temizlendi")


def get_cached_stages(patient_id, mongo_uri, db_name):
    """
    Hastanın stage verilerini cache'den getir, yoksa MongoDB'den yükle.
    
    Args:
        patient_id: Hasta ID'si
        mongo_uri: MongoDB bağlantı URI'si
        db_name: Veritabanı adı
    
    Returns:
        list: Stage verileri veya None
    """
    global _STAGES_CACHE
    
    # Cache'de var mı kontrol et
    if patient_id in _STAGES_CACHE:
        return _STAGES_CACHE[patient_id]
    
    # Yoksa MongoDB'den yükle
    client = MongoClient(mongo_uri)
    db = client[db_name]
    fs = gridfs.GridFS(db)
    
    stages_file = fs.find_one({
        "metadata.patient_id": patient_id,
        "metadata.data_type": "sleep_stages"
    })
    
    if not stages_file:
        client.close()
        return None
    
    import json
    stages_data = json.loads(stages_file.read().decode('utf-8'))
    
    # Cache'e ekle
    _STAGES_CACHE[patient_id] = stages_data
    
    client.close()
    return stages_data


def preload_all_stages(patient_ids=None, mongo_uri="mongodb://localhost:27017/",
                       db_name="psg_data"):
    """
    Tüm hastaların stage verilerini önceden cache'e yükle.
    
    Args:
        patient_ids: Hasta ID listesi (None ise tüm hastalar)
        mongo_uri: MongoDB bağlantı URI'si
        db_name: Veritabanı adı
    
    Returns:
        dict: Yükleme özeti
    """
    global _STAGES_CACHE
    
    print(f"\n{'='*70}")
    print(f"TÜM STAGE VERİLERİ CACHE'E YÜKLENİYOR")
    print(f"{'='*70}")
    
    client = MongoClient(mongo_uri)
    db = client[db_name]
    fs = gridfs.GridFS(db)
    
    # Hasta listesi verilmemişse tüm hastaları bul
    if patient_ids is None:
        patient_ids = set()
        for file in fs.find({"metadata.data_type": "sleep_stages"}):
            patient_ids.add(file.metadata['patient_id'])
        patient_ids = sorted(patient_ids)
    
    print(f"Toplam {len(patient_ids)} hasta bulundu")
    print(f"{'-'*70}")
    
    loaded_count = 0
    total_epochs = 0
    total_size_mb = 0
    
    import json
    
    for idx, patient_id in enumerate(patient_ids, 1):
        # Zaten cache'de varsa atla
        if patient_id in _STAGES_CACHE:
            print(f"[{idx}/{len(patient_ids)}] {patient_id}: Cache'de zaten mevcut ⊘")
            continue
        
        # MongoDB'den yükle
        stages_file = fs.find_one({
            "metadata.patient_id": patient_id,
            "metadata.data_type": "sleep_stages"
        })
        
        if not stages_file:
            print(f"[{idx}/{len(patient_ids)}] {patient_id}: Stage verisi bulunamadı ✗")
            continue
        
        stages_data = json.loads(stages_file.read().decode('utf-8'))
        
        # Cache'e ekle
        _STAGES_CACHE[patient_id] = stages_data
        
        # İstatistikler
        epoch_count = len(stages_data)
        size_bytes = len(json.dumps(stages_data))
        size_mb = size_bytes / (1024 * 1024)
        
        loaded_count += 1
        total_epochs += epoch_count
        total_size_mb += size_mb
        
        print(f"[{idx}/{len(patient_ids)}] {patient_id}: {epoch_count} epoch, "
              f"{size_mb:.2f} MB ✓")
    
    client.close()
    
    print(f"\n{'='*70}")
    print(f"CACHE YÜKLEME TAMAMLANDI")
    print(f"{'='*70}")
    print(f"Yüklenen hasta: {loaded_count}/{len(patient_ids)}")
    print(f"Toplam epoch: {total_epochs:,}")
    print(f"Toplam boyut: {total_size_mb:.2f} MB")
    print(f"Cache boyutu: {len(_STAGES_CACHE)} hasta")
    print(f"{'='*70}\n")
    
    return {
        'loaded': loaded_count,
        'total_patients': len(patient_ids),
        'total_epochs': total_epochs,
        'total_size_mb': total_size_mb,
        'cache_size': len(_STAGES_CACHE)
    }


def delete_patient_data(patient_id, mongo_uri="mongodb://localhost:27017/", 
                        db_name="psg_data", confirm=True):
    """
    Hastanın TÜM verilerini MongoDB'den siler (kanallar + uyku evreleri).
    
    Args:
        patient_id: Hasta ID'si
        mongo_uri: MongoDB bağlantı URI'si
        db_name: Veritabanı adı
        confirm: True ise kullanıcıdan onay ister
    
    Returns:
        dict: Silme işlemi özeti
    """
    client = MongoClient(mongo_uri)
    db = client[db_name]
    fs = gridfs.GridFS(db)
    
    # Hastaya ait tüm dosyaları bul
    patient_files = list(fs.find({"metadata.patient_id": patient_id}))
    
    if not patient_files:
        print(f"✗ Hasta bulunamadı: {patient_id}")
        client.close()
        return {
            'patient_id': patient_id,
            'status': 'not_found',
            'deleted_files': 0
        }
    
    # Dosya sayılarını hesapla
    channel_files = [f for f in patient_files if f.metadata.get('data_type') == 'raw_signal']
    stages_files = [f for f in patient_files if f.metadata.get('data_type') == 'sleep_stages']
    
    print(f"\n{'='*70}")
    print(f"Hasta ID: {patient_id}")
    print(f"{'='*70}")
    print(f"Kanal dosyaları: {len(channel_files)}")
    print(f"Uyku evresi dosyaları: {len(stages_files)}")
    print(f"Toplam silinecek dosya: {len(patient_files)}")
    
    # Kanal detaylarını göster
    if channel_files:
        print(f"\nKanallar:")
        for ch_file in channel_files:
            ch_meta = ch_file.metadata
            size_mb = ch_file.length / (1024 * 1024)
            print(f"  - {ch_meta['channel_name']} ({ch_meta['sample_frequency']} Hz, {size_mb:.2f} MB)")
    
    # Onay iste
    if confirm:
        print(f"\n⚠ UYARI: Bu işlem geri alınamaz!")
        response = input(f"\n'{patient_id}' hastasının TÜM verilerini silmek istediğinize emin misiniz? (evet/hayır): ")
        
        if response.lower() not in ['evet', 'yes', 'e', 'y']:
            print("İşlem iptal edildi.")
            client.close()
            return {
                'patient_id': patient_id,
                'status': 'cancelled',
                'deleted_files': 0
            }
    
    # Tüm dosyaları sil
    print(f"\nDosyalar siliniyor...")
    deleted_count = 0
    
    for file in patient_files:
        try:
            fs.delete(file._id)
            deleted_count += 1
            print(f"  ✓ Silindi: {file.filename}")
        except Exception as e:
            print(f"  ✗ Silinemedi: {file.filename} - {str(e)}")
    
    client.close()
    
    print(f"\n{'='*70}")
    print(f"✓ İşlem tamamlandı")
    print(f"Silinen dosya sayısı: {deleted_count}/{len(patient_files)}")
    print(f"{'='*70}")
    
    return {
        'patient_id': patient_id,
        'status': 'deleted',
        'deleted_files': deleted_count,
        'total_files': len(patient_files)
    }


def delete_patient_datas(patient_ids, mongo_uri="mongodb://localhost:27017/", 
                        db_name="psg_data", confirm=True):
    """
    Hastaların TÜM verilerini MongoDB'den siler (kanallar + uyku evreleri).
    
    Args:
        patient_ids: Hasta ID'leri
        mongo_uri: MongoDB bağlantı URI'si
        db_name: Veritabanı adı
        confirm: True ise kullanıcıdan onay ister
    
    Returns:
        dict: Silme işlemleri özeti
    """

    for patient_id in patient_ids:
        delete_patient_data(patient_id, mongo_uri, db_name, confirm)


def patient_exists_in_mongodb(patient_id, mongo_uri="mongodb://localhost:27017/", 
                               db_name="psg_data"):
    """
    Hastanın MongoDB'de olup olmadığını kontrol eder.
    
    Args:
        patient_id: Hasta ID'si
        mongo_uri: MongoDB bağlantı URI'si
        db_name: Veritabanı adı
    
    Returns:
        bool: Hasta varsa True, yoksa False
    """
    client = MongoClient(mongo_uri)
    db = client[db_name]
    fs = gridfs.GridFS(db)
    
    # Hasta ID'sine ait herhangi bir dosya var mı kontrol et
    exists = fs.find_one({"metadata.patient_id": patient_id}) is not None
    
    client.close()
    return exists


def process_patient_folders(root_folder, mongo_uri="mongodb://localhost:27017/", 
                            db_name="psg_data", skip_existing=True):
    """
    Klasörleri dolaşarak her hastanın verilerini MongoDB'ye aktarır.
    
    Klasör Yapısı:
        root_folder/
            patient_id_1/
                *.edf
                *.xml
            patient_id_2/
                *.edf
                *.xml
            ...
    
    Args:
        root_folder: Ana klasör yolu
        mongo_uri: MongoDB bağlantı URI'si
        db_name: Veritabanı adı
        skip_existing: True ise daha önce aktarılmış hastaları atlar
    
    Returns:
        dict: İşlem özeti
    """
    
    print(f"\n{'='*80}")
    print(f"TOPLU HASTA VERİSİ AKTARIMI")
    print(f"{'='*80}")
    print(f"Ana klasör: {root_folder}")
    print(f"Veritabanı: {db_name}")
    print(f"Mevcut hastaları atla: {skip_existing}")
    print(f"{'='*80}\n")
    
    # Ana klasördeki tüm alt klasörleri bul
    patient_folders = [f for f in os.listdir(root_folder) 
                      if os.path.isdir(os.path.join(root_folder, f))]
    
    print(f"Toplam {len(patient_folders)} hasta klasörü bulundu.\n")
    
    results = {
        'total': len(patient_folders),
        'processed': 0,
        'skipped': 0,
        'failed': 0,
        'details': []
    }
    
    for idx, patient_id in enumerate(sorted(patient_folders), 1):
        patient_folder = os.path.join(root_folder, patient_id)
        
        print(f"\n{'='*80}")
        print(f"[{idx}/{len(patient_folders)}] Hasta: {patient_id}")
        print(f"{'='*80}")
        
        # Hasta daha önce işlendi mi kontrol et
        if skip_existing and patient_exists_in_mongodb(patient_id, mongo_uri, db_name):
            print(f"⊘ ATLANDI - Bu hasta zaten MongoDB'de mevcut")
            results['skipped'] += 1
            results['details'].append({
                'patient_id': patient_id,
                'status': 'skipped',
                'reason': 'already_exists'
            })
            continue
        
        # EDF dosyasını bul
        edf_files = glob.glob(os.path.join(patient_folder, "*.edf"))
        if not edf_files:
            print(f"✗ HATA - EDF dosyası bulunamadı: {patient_folder}")
            results['failed'] += 1
            results['details'].append({
                'patient_id': patient_id,
                'status': 'failed',
                'reason': 'no_edf_file'
            })
            continue
        
        if len(edf_files) > 1:
            print(f"⚠ UYARI - Birden fazla EDF dosyası bulundu, ilki kullanılacak:")
            for ef in edf_files:
                print(f"    {os.path.basename(ef)}")
        
        edf_path = edf_files[0]
        print(f"EDF dosyası: {os.path.basename(edf_path)}")
        
        # XML dosyasını bul
        xml_files = glob.glob(os.path.join(patient_folder, "*.xml"))
        if not xml_files:
            print(f"✗ HATA - XML dosyası bulunamadı: {patient_folder}")
            results['failed'] += 1
            results['details'].append({
                'patient_id': patient_id,
                'status': 'failed',
                'reason': 'no_xml_file'
            })
            continue
        
        if len(xml_files) > 1:
            print(f"⚠ UYARI - Birden fazla XML dosyası bulundu, ilki kullanılacak:")
            for xf in xml_files:
                print(f"    {os.path.basename(xf)}")
        
        xml_path = xml_files[0]
        print(f"XML dosyası: {os.path.basename(xml_path)}")
        
        # MongoDB'ye aktar
        try:
            result = save_all_channels_to_mongodb(
                edf_path=edf_path,
                stages_path=xml_path,
                stages_format='xml',
                mongo_uri=mongo_uri,
                db_name=db_name,
                patient_id=patient_id
            )
            
            results['processed'] += 1
            results['details'].append({
                'patient_id': patient_id,
                'status': 'success',
                'channels': len(result['channels']),
                'epochs': len(result.get('sleep_stages', [])) if 'sleep_stages' in result else 'N/A'
            })
            
            print(f"\n✓ BAŞARILI - {patient_id} aktarıldı")
            
        except Exception as e:
            print(f"\n✗ HATA - {patient_id} aktarılırken hata oluştu:")
            print(f"    {str(e)}")
            results['failed'] += 1
            results['details'].append({
                'patient_id': patient_id,
                'status': 'failed',
                'reason': str(e)
            })
    
    # Özet rapor
    print(f"\n\n{'='*80}")
    print(f"AKTARIM ÖZETI")
    print(f"{'='*80}")
    print(f"Toplam klasör: {results['total']}")
    print(f"Başarılı: {results['processed']} ✓")
    print(f"Atlanan: {results['skipped']} ⊘")
    print(f"Başarısız: {results['failed']} ✗")
    print(f"{'='*80}\n")
    
    # Detaylı rapor
    if results['failed'] > 0:
        print("Başarısız aktarımlar:")
        for detail in results['details']:
            if detail['status'] == 'failed':
                print(f"  - {detail['patient_id']}: {detail['reason']}")
    
    if results['skipped'] > 0:
        print(f"\nAtlanan hastalar: {results['skipped']} hasta (zaten MongoDB'de mevcut)")
    
    return results


def parse_sleep_stages_xml(xml_path):
    """
    XML formatındaki uyku evresi etiketlerini parse eder.
    Sadece belirtilen uyku evrelerini (SLEEP-REM, SLEEP-S0, SLEEP-S1, SLEEP-S2, SLEEP-S3) alır.
    
    Args:
        xml_path: XML dosyasının yolu
    
    Returns:
        list: [
            {
                'epoch': int,
                'stage': str,
                'start_time': float,
                'duration': float,
                'onset': str
            },
            ...
        ]
    """
    print(f"XML dosyası parse ediliyor: {xml_path}")
    
    tree = ET.parse(xml_path)
    root = tree.getroot()
    
    # İstenen uyku evresi tipleri
    valid_event_types = {
        'SLEEP-REM': 'REM',
        'SLEEP-S0': 'W',
        'SLEEP-S1': 'N1',
        'SLEEP-S2': 'N2',
        'SLEEP-S3': 'N3'
    }
    
    stages = []
    epoch_counter = 0
    
    # Events elementini bul
    events_element = root.find('Events')
    if events_element is None:
        print("⚠ Events elementi bulunamadı")
        return stages
    
    # İlk eventi bul (referans zamanı için)
    first_event_time = None
    
    # Tüm Event elementlerini dolaş
    for event in events_element.findall('Event'):
        # Event tipini al
        type_elem = event.find('Type')
        if type_elem is None:
            continue
        
        event_type = type_elem.text.strip() if type_elem.text else None
        
        # Sadece istenen tipleri işle
        if event_type not in valid_event_types:
            continue
        
        # StartTime ve StopTime bilgilerini al
        start_time_elem = event.find('StartTime')
        stop_time_elem = event.find('StopTime')
        
        if start_time_elem is None or stop_time_elem is None:
            continue
        
        start_time_str = start_time_elem.text.strip() if start_time_elem.text else None
        stop_time_str = stop_time_elem.text.strip() if stop_time_elem.text else None
        
        if not start_time_str or not stop_time_str:
            continue
        
        try:
            # ISO format zamanlarını datetime'a çevir
            start_dt = datetime.fromisoformat(start_time_str.replace('Z', '+00:00'))
            stop_dt = datetime.fromisoformat(stop_time_str.replace('Z', '+00:00'))
            
            # İlk event zamanını referans olarak kaydet
            if first_event_time is None:
                first_event_time = start_dt
            
            # Başlangıçtan itibaren geçen süreyi hesapla (saniye)
            start_time = (start_dt - first_event_time).total_seconds()
            duration = (stop_dt - start_dt).total_seconds()
            
            # Uyku evresi etiketini normalize et
            stage = valid_event_types[event_type]
            
            stages.append({
                'epoch': epoch_counter,
                'stage': stage,
                'start_time': start_time,
                'duration': duration,
                'onset': start_time_str
            })
            
            epoch_counter += 1
            
        except (ValueError, AttributeError) as e:
            print(f"⚠ Event parse hatası: {str(e)}")
            continue
    
    # İstatistikleri yazdır
    if stages:
        stage_counts = {}
        total_duration = 0
        
        for stage_info in stages:
            stage = stage_info['stage']
            duration = stage_info['duration']
            stage_counts[stage] = stage_counts.get(stage, 0) + 1
            total_duration += duration
        
        print(f"✓ {len(stages)} epoch parse edildi")
        print(f"  Toplam süre: {total_duration:.2f} saniye ({total_duration/3600:.2f} saat)")
        print(f"  Uyku evresi dağılımı:")
        
        for stage, count in sorted(stage_counts.items()):
            percentage = (count / len(stages)) * 100
            print(f"    {stage}: {count} epoch ({percentage:.1f}%)")
    else:
        print(f"⚠ İstenen uyku evresi tiplerinde data bulunamadı")
    
    return stages


def parse_sleep_stages_edf(edf_path, annotation_channel=None):
    """
    EDF+ formatındaki annotation kanalından uyku evrelerini okur.
    
    Args:
        edf_path: EDF dosyasının yolu
        annotation_channel: Annotation kanalının indeksi (None ise otomatik bulur)
    
    Returns:
        list: [{epoch: int, stage: str, start_time: float, duration: float}, ...]
    """
    print(f"EDF dosyasından annotationlar okunuyor: {edf_path}")
    
    f = pyedflib.EdfReader(edf_path)
    
    try:
        # Annotation varsa oku
        annotations = f.readAnnotations()
        f.close()
        
        stages = []
        
        for i, (onset, duration, description) in enumerate(zip(
            annotations[0], annotations[1], annotations[2]
        )):
            # Uyku evresi annotationlarını filtrele
            stage = description.strip()
            
            # Yaygın uyku evresi etiketlerini normalize et
            stage_mapping = {
                'Sleep stage W': 'W',
                'Sleep stage R': 'REM',
                'Sleep stage N1': 'N1',
                'Sleep stage N2': 'N2',
                'Sleep stage N3': 'N3',
                'Sleep stage 1': 'N1',
                'Sleep stage 2': 'N2',
                'Sleep stage 3': 'N3',
                'Sleep stage 4': 'N3',
                'Wake': 'W',
                'REM sleep': 'REM',
            }
            
            stage = stage_mapping.get(stage, stage)
            
            # Sadece uyku evresi etiketlerini al
            if stage in ['W', 'REM', 'N1', 'N2', 'N3', 'N4']:
                stages.append({
                    'epoch': len(stages),
                    'stage': stage,
                    'start_time': float(onset),
                    'duration': float(duration)
                })
        
        print(f"✓ {len(stages)} epoch parse edildi")
        return stages
        
    except Exception as e:
        f.close()
        print(f"✗ EDF annotation okuma hatası: {str(e)}")
        return []


def save_all_channels_to_mongodb(edf_path, stages_path, stages_format='xml',
                                  mongo_uri="mongodb://localhost:27017/", 
                                  db_name="psg_data",
                                  patient_id=None):
    """
    EDF dosyasındaki TÜM kanalları ve uyku evresi etiketlerini GridFS'e kaydeder.
    Her kanal ayrı bir GridFS dosyası olarak saklanır, ancak uyku evreleri tüm kanallarda ortak.
    
    Args:
        edf_path: EDF dosyasının yolu
        stages_path: Uyku evresi etiketleri dosyasının yolu (XML veya EDF)
        stages_format: 'xml' veya 'edf'
        mongo_uri: MongoDB bağlantı URI'si
        db_name: Veritabanı adı
        patient_id: Hasta ID'si (opsiyonel, dosya isimlerini gruplamak için)
    
    Returns:
        dict: {
            'patient_id': str,
            'stages_id': ObjectId,  # Uyku evreleri dosya ID'si
            'channels': [
                {'channel_idx': int, 'channel_name': str, 'file_id': ObjectId, 'sample_freq': float},
                ...
            ]
        }
    """
    
    # Hasta ID'si yoksa dosya isminden oluştur
    if patient_id is None:
        patient_id = edf_path.split('/')[-1].split('.')[0]
    
    print(f"{'='*70}")
    print(f"Hasta ID: {patient_id}")
    print(f"{'='*70}")
    
    # Uyku evrelerini parse et
    print("\n[1/3] Uyku evreleri parse ediliyor...")
    if stages_format.lower() == 'xml':
        sleep_stages = parse_sleep_stages_xml(stages_path)
    elif stages_format.lower() == 'edf':
        sleep_stages = parse_sleep_stages_edf(stages_path)
    else:
        raise ValueError(f"Desteklenmeyen format: {stages_format}. 'xml' veya 'edf' olmalı.")
    
    # Uyku evresi istatistikleri
    stage_counts = {}
    for stage_info in sleep_stages:
        stage = stage_info['stage']
        stage_counts[stage] = stage_counts.get(stage, 0) + 1
    
    print(f"\nUyku evresi istatistikleri:")
    print(f"  Toplam epoch sayısı: {len(sleep_stages)}")
    for stage, count in sorted(stage_counts.items()):
        percentage = (count / len(sleep_stages)) * 100
        print(f"  {stage}: {count} epoch ({percentage:.1f}%)")
    
    # MongoDB'ye bağlan
    print(f"\n[2/3] MongoDB'ye bağlanılıyor...")
    client = MongoClient(mongo_uri)
    db = client[db_name]
    fs = gridfs.GridFS(db)
    
    # Önce uyku evrelerini ayrı bir dosya olarak kaydet
    stages_metadata = {
        "patient_id": patient_id,
        "data_type": "sleep_stages",
        "source_file": stages_path,
        "stages_format": stages_format,
        "total_epochs": len(sleep_stages),
        "epoch_duration": 30,
        "stage_counts": stage_counts,
        "upload_date": datetime.now()
    }
    
    # Uyku evrelerini JSON olarak kaydet
    stages_buffer = io.BytesIO()
    import json
    stages_json = json.dumps(sleep_stages, indent=2)
    stages_buffer.write(stages_json.encode('utf-8'))
    stages_buffer.seek(0)
    
    stages_id = fs.put(
        stages_buffer,
        filename=f"{patient_id}_sleep_stages.json",
        metadata=stages_metadata
    )
    
    print(f"✓ Uyku evreleri kaydedildi (ID: {stages_id})")
    
    # EDF dosyasını aç
    print(f"\n[3/3] EDF kanalları okunuyor ve kaydediliyor...")
    f = pyedflib.EdfReader(edf_path)
    
    channel_results = []
    
    try:
        n_channels = f.signals_in_file
        print(f"\nToplam kanal sayısı: {n_channels}")
        print(f"{'-'*70}")
        
        # Her kanalı oku ve kaydet
        for channel_idx in range(n_channels):
            signal_label = f.getLabel(channel_idx)
            sample_freq = f.getSampleFrequency(channel_idx)
            n_samples = f.getNSamples()[channel_idx]
            physical_min = f.getPhysicalMinimum(channel_idx)
            physical_max = f.getPhysicalMaximum(channel_idx)
            digital_min = f.getDigitalMinimum(channel_idx)
            digital_max = f.getDigitalMaximum(channel_idx)
            
            print(f"\nKanal {channel_idx + 1}/{n_channels}: {signal_label}")
            print(f"  Frekans: {sample_freq} Hz")
            print(f"  Örnek sayısı: {n_samples:,}")
            print(f"  Süre: {n_samples/sample_freq/3600:.2f} saat")
            
            # Sinyali oku
            signal_data = f.readSignal(channel_idx)
            data_size_mb = signal_data.nbytes / (1024 * 1024)
            print(f"  Boyut: {data_size_mb:.2f} MB")
            
            # Metadata hazırla - n_samples burada!
            channel_metadata = {
                "patient_id": patient_id,
                "channel_name": signal_label,
                "channel_index": int(channel_idx),
                "sample_frequency": float(sample_freq),
                "n_samples": int(n_samples),  # PARTIAL READ İÇİN GEREKLİ!
                "duration_hours": float(n_samples/sample_freq/3600),
                "physical_min": float(physical_min),
                "physical_max": float(physical_max),
                "digital_min": float(digital_min),
                "digital_max": float(digital_max),
                "data_type": "raw_signal",
                "dtype": str(signal_data.dtype),
                "source_file": edf_path,
                "upload_date": datetime.now(),
                # Uyku evreleri referansı
                "stages_id": stages_id,
                "has_sleep_stages": True
            }
            
            # NumPy array'i bytes'a çevir
            buffer = io.BytesIO()
            np.save(buffer, signal_data)
            buffer.seek(0)
            
            # GridFS'e kaydet
            file_id = fs.put(
                buffer,
                filename=f"{patient_id}_ch{channel_idx}_{signal_label}.npy",
                metadata=channel_metadata,
                chunk_size=255*1024
            )
            
            print(f"  ✓ Kaydedildi (ID: {file_id})")
            
            channel_results.append({
                'channel_idx': channel_idx,
                'channel_name': signal_label,
                'file_id': file_id,
                'sample_freq': sample_freq,
                'n_samples': n_samples,
                'data_size_mb': data_size_mb
            })
        
        # EDF dosyasını kapat
        f.close()
        
        # Bağlantıyı kapat
        client.close()
        
        # Özet
        print(f"\n{'='*70}")
        print(f"✓ TÜM VERİLER BAŞARIYLA KAYDEDİLDİ!")
        print(f"{'='*70}")
        print(f"Hasta ID: {patient_id}")
        print(f"Uyku evreleri ID: {stages_id}")
        print(f"Toplam kanal sayısı: {len(channel_results)}")
        total_size = sum(ch['data_size_mb'] for ch in channel_results)
        print(f"Toplam veri boyutu: {total_size:.2f} MB")
        print(f"Veritabanı: {db_name}")
        
        return {
            'patient_id': patient_id,
            'stages_id': stages_id,
            'channels': channel_results
        }
        
    except Exception as e:
        f.close()
        client.close()
        print(f"\n✗ Hata oluştu: {str(e)}")
        raise


def read_patient_data(patient_id, mongo_uri="mongodb://localhost:27017/", 
                      db_name="psg_data", channel_name=None):
    """
    Hasta verilerini MongoDB'den okur.
    
    Args:
        patient_id: Hasta ID'si
        mongo_uri: MongoDB bağlantı URI'si
        db_name: Veritabanı adı
        channel_name: Belirli bir kanal okumak için (None ise tüm kanalları listeler)
    
    Returns:
        dict: Hasta verisi
    """
    client = MongoClient(mongo_uri)
    db = client[db_name]
    fs = gridfs.GridFS(db)
    
    # Uyku evrelerini bul
    stages_file = fs.find_one({
        "metadata.patient_id": patient_id,
        "metadata.data_type": "sleep_stages"
    })
    
    if not stages_file:
        print(f"✗ Hasta {patient_id} için uyku evreleri bulunamadı")
        client.close()
        return None
    
    # Uyku evrelerini oku
    import json
    stages_data = json.loads(stages_file.read().decode('utf-8'))
    
    # Kanalları bul
    channel_files = list(fs.find({
        "metadata.patient_id": patient_id,
        "metadata.data_type": "raw_signal"
    }))
    
    print(f"\nHasta ID: {patient_id}")
    print(f"Uyku evreleri: {len(stages_data)} epoch")
    print(f"Toplam kanal sayısı: {len(channel_files)}")
    print(f"\nMevcut kanallar:")
    
    for ch_file in channel_files:
        ch_meta = ch_file.metadata
        print(f"  - {ch_meta['channel_name']} ({ch_meta['sample_frequency']} Hz)")
    
    # Belirli bir kanal istendiyse
    if channel_name:
        print(f"\nKanal okunuyor: {channel_name}")
        for ch_file in channel_files:
            if ch_file.metadata['channel_name'] == channel_name:
                buffer = io.BytesIO(ch_file.read())
                signal_data = np.load(buffer)
                
                print(f"✓ Kanal yüklendi: {len(signal_data):,} örnek")
                
                client.close()
                return {
                    'patient_id': patient_id,
                    'channel_name': channel_name,
                    'signal_data': signal_data,
                    'metadata': ch_file.metadata,
                    'sleep_stages': stages_data
                }
        
        print(f"✗ Kanal '{channel_name}' bulunamadı")
    
    client.close()
    return {
        'patient_id': patient_id,
        'available_channels': [ch.metadata['channel_name'] for ch in channel_files],
        'sleep_stages': stages_data
    }


def list_all_patients(mongo_uri="mongodb://localhost:27017/", 
                      db_name="psg_data",
                      show_channels_info=False):
    """
    Veritabanındaki tüm hastaları listeler.
    """
    client = MongoClient(mongo_uri)
    db = client[db_name]
    fs = gridfs.GridFS(db)
    
    # Tüm hasta ID'lerini bul
    patient_ids = set()
    for file in fs.find({"metadata.patient_id": {"$exists": True}}):
        patient_ids.add(file.metadata['patient_id'])
    
    print(f"\nVeritabanında {len(patient_ids)} hasta bulundu:")
    print(f"{'='*70}")
    
    for patient_id in sorted(patient_ids):
        # Hasta bilgilerini al
        channel_files = list(fs.find({
            "metadata.patient_id": patient_id,
            "metadata.data_type": "raw_signal"
        }))
        channel_count = len(channel_files)

        if channel_files and show_channels_info:
            print(f"\nKanallar:")
            for ch_file in channel_files:
                ch_meta = ch_file.metadata
                size_mb = ch_file.length / (1024 * 1024)
                print(f"  - {ch_meta['channel_name']} ({ch_meta['sample_frequency']} Hz, {size_mb:.2f} MB)")
        
        stages_file = fs.find_one({
            "metadata.patient_id": patient_id,
            "metadata.data_type": "sleep_stages"
        })
        
        epoch_count = stages_file.metadata.get('total_epochs', 0) if stages_file else 0
        
        print(f"\nHasta ID: {patient_id}")
        print(f"  Kanal sayısı: {channel_count}")
        print(f"  Epoch sayısı: {epoch_count}")
        
        if stages_file:
            stage_counts = stages_file.metadata.get('stage_counts', {})
            print(f"  Uyku evreleri: {', '.join([f'{k}={v}' for k, v in stage_counts.items()])}")
    
    client.close()


def extract_epoch_range_data_v2(patient_id, channel_name, start_epoch, end_epoch,
                             mongo_uri="mongodb://localhost:27017/",
                             db_name="psg_data",
                             save_to_file=False,
                             output_file=None,
                             output_format='npy'):
    """
    Belirtilen hasta, kanal ve epoch aralığına ait ham veriyi çeker.
    OPTIMIZE: 
    - Signal verisi: Partial read (GridFS chunks)
    - Stage verisi: Session cache (ilk yüklemeden sonra bellekte)
    
    
    Args:
        patient_id: Hasta ID'si
        channel_name: Kanal adı (örn: "E1", "C3-A2")
        start_epoch: Başlangıç epoch numarası (dahil)
        end_epoch: Bitiş epoch numarası (dahil)
        mongo_uri: MongoDB bağlantı URI'si
        db_name: Veritabanı adı
        save_to_file: True ise dosyaya kaydeder
        output_file: Çıktı dosyasının yolu (None ise otomatik oluşturulur)
        output_format: Dosya formatı ('npy', 'csv', 'txt')
    
    Returns:
        dict: Hasta, kanal ve sinyal bilgilerini içeren sözlük
    """
    
    print(f"\n{'='*70}")
    print(f"VERİ ÇEKME İŞLEMİ (OPTİMİZE - PARTIAL READ)")
    print(f"{'='*70}")
    print(f"Hasta ID: {patient_id}")
    print(f"Kanal: {channel_name}")
    print(f"Epoch Aralığı: {start_epoch} - {end_epoch} (toplam {end_epoch - start_epoch + 1} epoch)")
    
    # MongoDB'ye bağlan
    client = MongoClient(mongo_uri)
    db = client[db_name]
    fs = gridfs.GridFS(db)
    
    try:
        # 1. Uyku evrelerini cache'den al
        print(f"\n[1/4] Uyku evreleri okunuyor...")
        
        all_stages = get_cached_stages(patient_id, mongo_uri, db_name)
        
        if all_stages is None:
            print(f"✗ Hasta {patient_id} için uyku evreleri bulunamadı")
            client.close()
            return None
        
        # Cache'den mi geldi kontrol et
        if patient_id in _STAGES_CACHE and _STAGES_CACHE[patient_id] is all_stages:
            print(f"✓ {len(all_stages)} epoch bulundu (cache'den alındı - MongoDB'ye gidilmedi)")
        else:
            print(f"✓ {len(all_stages)} epoch bulundu (MongoDB'den yüklendi ve cache'lendi)")
        
        total_epochs = len(all_stages)
        
        # Epoch aralığını kontrol et
        if start_epoch < 0 or end_epoch >= total_epochs:
            print(f"✗ Geçersiz epoch aralığı! (Mevcut: 0-{total_epochs-1})")
            client.close()
            return None
        
        if start_epoch > end_epoch:
            print(f"✗ Başlangıç epoch'u bitiş epoch'undan büyük olamaz!")
            client.close()
            return None
        
        # İstenen epoch'ların zaman bilgilerini al (bellekte filtreleme - çok hızlı)
        stages_data = all_stages[start_epoch:end_epoch+1]
        start_time = stages_data[0]['start_time']
        end_time = stages_data[-1]['start_time'] + stages_data[-1]['duration']
        duration = end_time - start_time
        
        print(f"  Zaman aralığı: {start_time:.2f}s - {end_time:.2f}s (süre: {duration:.2f}s)")
        
        # 2. Kanal dosyasını bul (metadata'yı al)
        print(f"\n[2/4] Kanal metadata'sı alınıyor...")
        channel_file = fs.find_one({
            "metadata.patient_id": patient_id,
            "metadata.channel_name": channel_name,
            "metadata.data_type": "raw_signal"
        })
        
        if not channel_file:
            print(f"✗ Kanal '{channel_name}' bulunamadı")
            
            # Mevcut kanalları göster
            available_channels = []
            for ch_file in fs.find({
                "metadata.patient_id": patient_id,
                "metadata.data_type": "raw_signal"
            }):
                available_channels.append(ch_file.metadata['channel_name'])
            
            if available_channels:
                print(f"\nMevcut kanallar:")
                for ch in sorted(available_channels):
                    print(f"  - {ch}")
            
            client.close()
            return None
        
        # Kanal metadata'sını al
        ch_metadata = channel_file.metadata
        sample_freq = ch_metadata['sample_frequency']
        n_samples = ch_metadata.get('n_samples', 0)
        
        print(f"✓ Kanal bulundu")
        print(f"  Örnekleme frekansı: {sample_freq} Hz")
        print(f"  Toplam örnek sayısı: {n_samples:,}")
        
        # 3. Okunacak byte aralığını hesapla
        print(f"\n[3/4] Okunacak veri aralığı hesaplanıyor...")
        
        # Zaman bilgisini örnek indeksine çevir
        start_sample = int(start_time * sample_freq)
        end_sample = int(end_time * sample_freq)
        
        # Sınırları kontrol et
        start_sample = max(0, start_sample)
        end_sample = min(n_samples, end_sample)
        
        # NumPy array float64 = 8 byte per sample
        bytes_per_sample = 8
        
        # NPY dosya formatı: 128 byte header + veri
        npy_header_size = 128
        
        # Okunacak byte pozisyonları
        start_byte = npy_header_size + (start_sample * bytes_per_sample)
        end_byte = npy_header_size + (end_sample * bytes_per_sample)
        bytes_to_read = end_byte - start_byte
        
        print(f"✓ Hesaplama tamamlandı")
        print(f"  Örnek aralığı: {start_sample:,} - {end_sample:,}")
        print(f"  Byte aralığı: {start_byte:,} - {end_byte:,}")
        print(f"  Okunacak veri: {bytes_to_read / (1024*1024):.2f} MB")
        
        # 4. GridFS'den sadece gerekli bölümü oku (PARTIAL READ)
        print(f"\n[4/4] Veri kısmi okuma ile çekiliyor...")
        
        # GridFS'den sadece belirtilen byte aralığını oku
        file_id = channel_file._id
        chunks_collection = db['fs.chunks']
        
        # Chunk boyutunu al (varsayılan 255KB)
        chunk_size = channel_file.chunk_size
        
        # Hangi chunk'ları okuyacağımızı hesapla
        start_chunk_n = start_byte // chunk_size
        end_chunk_n = (end_byte - 1) // chunk_size
        
        print(f"  Chunk boyutu: {chunk_size / 1024:.0f} KB")
        print(f"  Okunacak chunk'lar: {start_chunk_n} - {end_chunk_n} (toplam {end_chunk_n - start_chunk_n + 1} chunk)")
        
        # İlgili chunk'ları çek
        chunks = chunks_collection.find({
            'files_id': file_id,
            'n': {'$gte': start_chunk_n, '$lte': end_chunk_n}
        }).sort('n', 1)
        
        # Chunk'ları birleştir
        combined_data = b''
        for chunk in chunks:
            combined_data += chunk['data']
        
        # Chunk içindeki offset'i hesapla
        offset_in_first_chunk = start_byte % chunk_size
        offset_in_last_chunk = end_byte % chunk_size
        
        # Gereksiz baş ve son kısımları çıkar
        if start_chunk_n == end_chunk_n:
            # Aynı chunk içindeyse
            combined_data = combined_data[offset_in_first_chunk:offset_in_first_chunk + bytes_to_read]
        else:
            # Farklı chunk'lardaysa
            combined_data = combined_data[offset_in_first_chunk:]
            if offset_in_last_chunk > 0:
                total_length = len(combined_data)
                excess = total_length - bytes_to_read
                if excess > 0:
                    combined_data = combined_data[:-excess]
        
        # Byte array'i numpy array'e çevir
        extracted_signal = np.frombuffer(combined_data, dtype=np.float64)
        
        print(f"✓ Veri çıkarıldı")
        print(f"  Çıkarılan örnek sayısı: {len(extracted_signal):,}")
        print(f"  Veri boyutu: {extracted_signal.nbytes / (1024*1024):.2f} MB")
        print(f"  Bellek tasarrufu: %{(1 - bytes_to_read / (n_samples * bytes_per_sample)) * 100:.1f}")
        
        # Sonuç dictionary'si
        result = {
            'patient_id': patient_id,
            'channel_name': channel_name,
            'start_epoch': start_epoch,
            'end_epoch': end_epoch,
            'epoch_count': end_epoch - start_epoch + 1,
            'signal_data': extracted_signal,
            'sample_frequency': sample_freq,
            'start_time': start_time,
            'end_time': end_time,
            'duration': duration,
            'start_sample': start_sample,
            'end_sample': end_sample
        }
        
        # 5. Dosyaya kaydet (istenirse)
        if save_to_file:
            print(f"\n[5/5] Dosyaya kaydediliyor...")
            
            # Dosya adı oluştur
            if output_file is None:
                output_file = f"{patient_id}_{channel_name}_epoch{start_epoch}-{end_epoch}.{output_format}"
            
            if output_format == 'npy':
                np.save(output_file, extracted_signal)
                print(f"✓ NumPy formatında kaydedildi: {output_file}")
            
            elif output_format == 'csv':
                np.savetxt(output_file, extracted_signal, delimiter=',', fmt='%.6f')
                print(f"✓ CSV formatında kaydedildi: {output_file}")
            
            elif output_format == 'txt':
                np.savetxt(output_file, extracted_signal, fmt='%.6f')
                print(f"✓ TXT formatında kaydedildi: {output_file}")
            
            else:
                print(f"✗ Desteklenmeyen format: {output_format}")
                output_file = None
            
            result['output_file'] = output_file
        
        client.close()
        
        print(f"\n{'='*70}")
        print(f"✓ İŞLEM TAMAMLANDI")
        print(f"{'='*70}\n")
        
        return result
        
    except Exception as e:
        print(f"\n✗ Hata oluştu: {str(e)}")
        import traceback
        traceback.print_exc()
        client.close()
        return None


def extract_epoch_range_whole_data_for_validation(): 
    # DONT TOUCH THIS SECTION BECAUSE THIS WILL USE FOR VALIDATION
    print("")
    # def extract_epoch_range_data(patient_id, channel_name, start_epoch, end_epoch,
    #                              mongo_uri="mongodb://localhost:27017/",
    #                              db_name="psg_data",
    #                              save_to_file=False,
    #                              output_file=None,
    #                              output_format='npy'):
    #     """
    #     Belirtilen hasta, kanal ve epoch aralığına ait ham veriyi çeker.
    #     Önce tüm kanal çekiliyor, sonra slice yaılıyor. Validation için bırakıldı.
        
    #     Args:
    #         patient_id: Hasta ID'si
    #         channel_name: Kanal adı (örn: "E1", "C3-A2")
    #         start_epoch: Başlangıç epoch numarası (dahil)
    #         end_epoch: Bitiş epoch numarası (dahil)
    #         mongo_uri: MongoDB bağlantı URI'si
    #         db_name: Veritabanı adı
    #         save_to_file: True ise dosyaya kaydeder
    #         output_file: Çıktı dosyasının yolu (None ise otomatik oluşturulur)
    #         output_format: Dosya formatı ('npy', 'csv', 'txt')
        
    #     Returns:
    #         dict: {
    #             'patient_id': str,
    #             'channel_name': str,
    #             'start_epoch': int,
    #             'end_epoch': int,
    #             'epoch_count': int,
    #             'signal_data': np.array,
    #             'sample_frequency': float,
    #             'start_time': float,
    #             'end_time': float,
    #             'duration': float,
    #             'output_file': str (eğer save_to_file=True ise)
    #         }
    #     """
        
    #     print(f"\n{'='*70}")
    #     print(f"VERİ ÇEKME İŞLEMİ")
    #     print(f"{'='*70}")
    #     print(f"Hasta ID: {patient_id}")
    #     print(f"Kanal: {channel_name}")
    #     print(f"Epoch Aralığı: {start_epoch} - {end_epoch} (toplam {end_epoch - start_epoch + 1} epoch)")
        
    #     # MongoDB'ye bağlan
    #     client = MongoClient(mongo_uri)
    #     db = client[db_name]
    #     fs = gridfs.GridFS(db)
        
    #     try:
    #         # 1. Uyku evrelerini çek
    #         print(f"\n[1/3] Uyku evreleri okunuyor...")
    #         stages_file = fs.find_one({
    #             "metadata.patient_id": patient_id,
    #             "metadata.data_type": "sleep_stages"
    #         })
            
    #         if not stages_file:
    #             print(f"✗ Hasta {patient_id} için uyku evreleri bulunamadı")
    #             client.close()
    #             return None
            
    #         import json
    #         stages_data = json.loads(stages_file.read().decode('utf-8'))
    #         total_epochs = len(stages_data)
            
    #         print(f"✓ Toplam {total_epochs} epoch bulundu")
            
    #         # Epoch aralığını kontrol et
    #         if start_epoch < 0 or end_epoch >= total_epochs:
    #             print(f"✗ Geçersiz epoch aralığı! (Mevcut: 0-{total_epochs-1})")
    #             client.close()
    #             return None
            
    #         if start_epoch > end_epoch:
    #             print(f"✗ Başlangıç epoch'u bitiş epoch'undan büyük olamaz!")
    #             client.close()
    #             return None
            
    #         # İstenen epoch'ların zaman bilgilerini al
    #         start_time = stages_data[start_epoch]['start_time']
    #         end_time = stages_data[end_epoch]['start_time'] + stages_data[end_epoch]['duration']
    #         duration = end_time - start_time
            
    #         print(f"  Zaman aralığı: {start_time:.2f}s - {end_time:.2f}s (süre: {duration:.2f}s)")
            
    #         # 2. Kanal verisini çek
    #         print(f"\n[2/3] Kanal verisi okunuyor...")
    #         channel_file = fs.find_one({
    #             "metadata.patient_id": patient_id,
    #             "metadata.channel_name": channel_name,
    #             "metadata.data_type": "raw_signal"
    #         })
            
    #         if not channel_file:
    #             print(f"✗ Kanal '{channel_name}' bulunamadı")
                
    #             # Mevcut kanalları göster
    #             available_channels = []
    #             for ch_file in fs.find({
    #                 "metadata.patient_id": patient_id,
    #                 "metadata.data_type": "raw_signal"
    #             }):
    #                 available_channels.append(ch_file.metadata['channel_name'])
                
    #             if available_channels:
    #                 print(f"\nMevcut kanallar:")
    #                 for ch in sorted(available_channels):
    #                     print(f"  - {ch}")
                
    #             client.close()
    #             return None
            
    #         # Kanal metadata'sını al
    #         ch_metadata = channel_file.metadata
    #         sample_freq = ch_metadata['sample_frequency']
            
    #         print(f"✓ Kanal bulundu")
    #         print(f"  Örnekleme frekansı: {sample_freq} Hz")
            
    #         # Tam sinyali oku
    #         buffer = io.BytesIO(channel_file.read())
    #         full_signal = np.load(buffer)
            
    #         print(f"  Toplam örnek sayısı: {len(full_signal):,}")
            
    #         # 3. Epoch aralığına karşılık gelen örnekleri çıkar
    #         print(f"\n[3/3] Epoch aralığı kesiliyor...")
            
    #         # Zaman bilgisini örnek indeksine çevir
    #         start_sample = int(start_time * sample_freq)
    #         end_sample = int(end_time * sample_freq)
            
    #         # Sınırları kontrol et
    #         start_sample = max(0, start_sample)
    #         end_sample = min(len(full_signal), end_sample)
            
    #         # İlgili bölümü kes
    #         extracted_signal = full_signal[start_sample:end_sample]
            
    #         print(f"✓ Veri çıkarıldı")
    #         print(f"  Örnek aralığı: {start_sample:,} - {end_sample:,}")
    #         print(f"  Çıkarılan örnek sayısı: {len(extracted_signal):,}")
    #         print(f"  Veri boyutu: {extracted_signal.nbytes / (1024*1024):.2f} MB")
            
    #         # Sonuç dictionary'si
    #         result = {
    #             'patient_id': patient_id,
    #             'channel_name': channel_name,
    #             'start_epoch': start_epoch,
    #             'end_epoch': end_epoch,
    #             'epoch_count': end_epoch - start_epoch + 1,
    #             'signal_data': extracted_signal,
    #             'sample_frequency': sample_freq,
    #             'start_time': start_time,
    #             'end_time': end_time,
    #             'duration': duration,
    #             'start_sample': start_sample,
    #             'end_sample': end_sample
    #         }
            
    #         # 4. Dosyaya kaydet (istenirse)
    #         if save_to_file:
    #             print(f"\n[4/4] Dosyaya kaydediliyor...")
                
    #             # Dosya adı oluştur
    #             if output_file is None:
    #                 output_file = f"{patient_id}_{channel_name}_epoch{start_epoch}-{end_epoch}.{output_format}"
                
    #             if output_format == 'npy':
    #                 np.save(output_file, extracted_signal)
    #                 print(f"✓ NumPy formatında kaydedildi: {output_file}")
                
    #             elif output_format == 'csv':
    #                 np.savetxt(output_file, extracted_signal, delimiter=',', fmt='%.6f')
    #                 print(f"✓ CSV formatında kaydedildi: {output_file}")
                
    #             elif output_format == 'txt':
    #                 np.savetxt(output_file, extracted_signal, fmt='%.6f')
    #                 print(f"✓ TXT formatında kaydedildi: {output_file}")
                
    #             else:
    #                 print(f"✗ Desteklenmeyen format: {output_format}")
    #                 output_file = None
                
    #             result['output_file'] = output_file
            
    #         client.close()
            
    #         print(f"\n{'='*70}")
    #         print(f"✓ İŞLEM TAMAMLANDI")
    #         print(f"{'='*70}\n")
            
    #         return result
            
    #     except Exception as e:
    #         print(f"\n✗ Hata oluştu: {str(e)}")
    #         import traceback
    #         traceback.print_exc()
    #         client.close()
    #         return None




In [2]:
# Kullanım örneği
if __name__ == "__main__":
    
    # ====================================================================
    # TOPLU AKTARIM - Tüm hasta klasörlerini işle
    # ====================================================================
    
    # Ana klasör yolumuz
    root_folder = "/mnt/data1/data/owndata/2.uncompressed_data/1.first_phase"
    
    # Toplu aktarımı başlat
    results = process_patient_folders(
        root_folder=root_folder,
        mongo_uri="mongodb://localhost:27017/",
        db_name="psg_data",
        skip_existing=True  # Var olanları atla
    )

    preload_all_stages(
        mongo_uri="mongodb://localhost:27017/",
        db_name="psg_data"
    )

    # Cache'i temizler (gerekirse)
    # clear_stages_cache()

    



TOPLU HASTA VERİSİ AKTARIMI
Ana klasör: /mnt/data1/data/owndata/2.uncompressed_data/1.first_phase
Veritabanı: psg_data
Mevcut hastaları atla: True

Toplam 17 hasta klasörü bulundu.


[1/17] Hasta: 22cd54f1-5d96-4b12-a392-d5ddada9d8d3
⊘ ATLANDI - Bu hasta zaten MongoDB'de mevcut

[2/17] Hasta: 28ba35f0-a068-4eeb-a51a-f4c45b4c0843
⊘ ATLANDI - Bu hasta zaten MongoDB'de mevcut

[3/17] Hasta: 2d42885b-ba49-4c21-9e8f-4963349f252c
⊘ ATLANDI - Bu hasta zaten MongoDB'de mevcut

[4/17] Hasta: 3da2681a-5991-449c-80a4-52cf3257db02
⊘ ATLANDI - Bu hasta zaten MongoDB'de mevcut

[5/17] Hasta: 441425cd-4ee5-45b6-a33a-fd7de892d61f
⊘ ATLANDI - Bu hasta zaten MongoDB'de mevcut

[6/17] Hasta: 4f680835-27cb-422f-b254-1d2994ff548a
⊘ ATLANDI - Bu hasta zaten MongoDB'de mevcut

[7/17] Hasta: 596ce269-a11b-4dce-859e-9b4f25428033
⊘ ATLANDI - Bu hasta zaten MongoDB'de mevcut

[8/17] Hasta: 66d496f0-3092-48db-801e-ae81fb623131
⊘ ATLANDI - Bu hasta zaten MongoDB'de mevcut

[9/17] Hasta: 6b19b899-e4b2-4465-93d7-ab

In [5]:
# HİPNOGRAM

def create_hypnogram(patient_id, mongo_uri="mongodb://localhost:27017/",
                     db_name="psg_data", output_folder="hypnograms",
                     figsize=(16, 6), dpi=150, _stages_cache=None):
    """
    Belirtilen hasta için hipnogram grafiği oluşturur ve kaydeder.
    Cache sistemini kullanarak MongoDB'ye gereksiz bağlantıları önler.
    
    Args:
        patient_id: Hasta ID'si
        mongo_uri: MongoDB bağlantı URI'si
        db_name: Veritabanı adı
        output_folder: Grafiklerin kaydedileceği klasör
        figsize: Grafik boyutu (genişlik, yükseklik)
        dpi: Grafik çözünürlüğü
        _stages_cache: Stage cache referansı (opsiyonel, dışarıdan inject edilebilir)
    
    Returns:
        str: Kaydedilen dosyanın yolu veya None
    """
    
    print(f"\n{'='*70}")
    print(f"HİPNOGRAM OLUŞTURULUYOR: {patient_id}")
    print(f"{'='*70}")
    
    try:
        # Cache'den veya get_cached_stages ile al
        if _stages_cache is not None and patient_id in _stages_cache:
            stages_data = _stages_cache[patient_id]
            print(f"✓ {len(stages_data)} epoch yüklendi (cache'den - MongoDB'ye gidilmedi)")
        else:
            # get_cached_stages fonksiyonunu kullan (otomatik cache yönetimi)
            stages_data = get_cached_stages(patient_id, mongo_uri, db_name)
            
            if stages_data is None:
                print(f"✗ Hasta {patient_id} için uyku evreleri bulunamadı")
                return None
            
            print(f"✓ {len(stages_data)} epoch yüklendi (get_cached_stages ile)")
        
        # Uyku evrelerini sayısal değerlere dönüştür
        stage_map = {
            'W': 0,      # Uyanık
            'REM': -1,   # REM uykusu
            'N1': -2,    # N1 (hafif uyku)
            'N2': -3,    # N2 (orta uyku)
            'N3': -4     # N3 (derin uyku)
        }
        
        stage_labels = {
            0: 'Wake',
            -1: 'REM',
            -2: 'N1',
            -3: 'N2',
            -4: 'N3'
        }
        
        # Renk paleti
        stage_colors = {
            0: '#FF6B6B',    # Kırmızı - Uyanık
            -1: '#4ECDC4',   # Turkuaz - REM
            -2: '#95E1D3',   # Açık yeşil - N1
            -3: '#45B7D1',   # Mavi - N2
            -4: '#2C3E50'    # Koyu mavi - N3
        }
        
        # Veriyi hazırla
        times = []
        stages = []
        
        for epoch_data in stages_data:
            stage_name = epoch_data['stage']
            start_time = epoch_data['start_time']
            duration = epoch_data['duration']
            
            if stage_name in stage_map:
                times.append(start_time / 3600)  # Saate çevir
                stages.append(stage_map[stage_name])
        
        times = np.array(times)
        stages = np.array(stages)
        
        # İstatistikleri hesapla
        total_time_hours = times[-1] if len(times) > 0 else 0
        stage_stats = {}
        
        for stage_name, stage_value in stage_map.items():
            count = np.sum(stages == stage_value)
            duration_hours = count * 30 / 3600  # 30 saniyelik epoch'lar
            percentage = (count / len(stages)) * 100 if len(stages) > 0 else 0
            stage_stats[stage_name] = {
                'count': count,
                'duration_hours': duration_hours,
                'percentage': percentage
            }
        
        # Grafik oluştur
        fig, ax = plt.subplots(figsize=figsize, dpi=dpi)
        
        # Her epoch için dikdörtgen çiz
        epoch_duration = 30 / 3600  # 30 saniye = saat cinsinden
        
        for i in range(len(times)):
            color = stage_colors[stages[i]]
            rect = mpatches.Rectangle(
                (times[i], stages[i] - 0.4),
                epoch_duration,
                0.8,
                facecolor=color,
                edgecolor='none',
                alpha=0.9
            )
            ax.add_patch(rect)
        
        # Eksen ayarları
        ax.set_xlim(0, max(times[-1], 8))  # En az 8 saat göster
        ax.set_ylim(-4.5, 0.5)
        
        # Y ekseni etiketleri
        ax.set_yticks([0, -1, -2, -3, -4])
        ax.set_yticklabels(['Wake', 'REM', 'N1', 'N2', 'N3'])
        
        # X ekseni (saat formatında)
        ax.set_xlabel('Zaman (Saat)', fontsize=12, fontweight='bold')
        ax.set_ylabel('Uyku Evresi', fontsize=12, fontweight='bold')
        
        # Grid
        ax.grid(True, axis='x', alpha=0.3, linestyle='--')
        ax.grid(True, axis='y', alpha=0.2)
        
        # Başlık
        title = f'Hipnogram - Hasta: {patient_id}\n'
        title += f'Toplam Kayıt Süresi: {total_time_hours:.1f} saat | '
        title += f'Toplam Epoch: {len(stages_data)}'
        ax.set_title(title, fontsize=14, fontweight='bold', pad=20)
        
        # İstatistik kutusu
        stats_text = 'Uyku Evresi Dağılımı:\n'
        stats_text += '-' * 30 + '\n'
        
        for stage_name in ['W', 'REM', 'N1', 'N2', 'N3']:
            stats = stage_stats[stage_name]
            stats_text += f'{stage_labels[stage_map[stage_name]]}: '
            stats_text += f'{stats["duration_hours"]:.1f}h ({stats["percentage"]:.1f}%)\n'
        
        # İstatistik kutusunu sağ üst köşeye ekle
        props = dict(boxstyle='round', facecolor='white', alpha=0.9, edgecolor='gray')
        ax.text(0.98, 0.97, stats_text,
                transform=ax.transAxes,
                fontsize=10,
                verticalalignment='top',
                horizontalalignment='right',
                bbox=props,
                family='monospace')
        
        # Legend
        legend_elements = [
            mpatches.Patch(facecolor=stage_colors[0], label='Wake (Uyanık)', alpha=0.9),
            mpatches.Patch(facecolor=stage_colors[-1], label='REM Sleep', alpha=0.9),
            mpatches.Patch(facecolor=stage_colors[-2], label='N1 (Hafif Uyku)', alpha=0.9),
            mpatches.Patch(facecolor=stage_colors[-3], label='N2 (Orta Uyku)', alpha=0.9),
            mpatches.Patch(facecolor=stage_colors[-4], label='N3 (Derin Uyku)', alpha=0.9)
        ]
        ax.legend(handles=legend_elements, loc='upper left', 
                 framealpha=0.9, fontsize=9)
        
        plt.tight_layout()
        
        # Klasörü oluştur
        os.makedirs(output_folder, exist_ok=True)
        
        # Dosya adı ve yolu
        output_path = os.path.join(output_folder, f"{patient_id}_hypnogram.png")
        
        # Kaydet
        plt.savefig(output_path, dpi=dpi, bbox_inches='tight', 
                   facecolor='white', edgecolor='none')
        plt.close(fig)
        
        print(f"✓ Hipnogram kaydedildi: {output_path}")
        print(f"  Boyut: {figsize[0]}x{figsize[1]} inch, {dpi} DPI")
        print(f"{'='*70}\n")
        
        return output_path
        
    except Exception as e:
        print(f"✗ Hata oluştu: {str(e)}")
        import traceback
        traceback.print_exc()
        return None


def create_all_hypnograms(mongo_uri="mongodb://localhost:27017/",
                          db_name="psg_data",
                          output_folder="hypnograms",
                          patient_ids=None,
                          figsize=(16, 6),
                          dpi=150,
                          use_cache=True):
    """
    Veritabanındaki tüm hastalar için hipnogram oluşturur.
    Cache sistemini kullanarak performanslı çalışır.
    
    Args:
        mongo_uri: MongoDB bağlantı URI'si
        db_name: Veritabanı adı
        output_folder: Grafiklerin kaydedileceği klasör
        patient_ids: İşlenecek hasta ID listesi (None ise tümü)
        figsize: Grafik boyutu
        dpi: Grafik çözünürlük
        use_cache: True ise global cache'i kullanır, False ise her seferinde DB'den okur
    
    Returns:
        dict: İşlem özeti
    """
    
    print(f"\n{'='*70}")
    print(f"TOPLU HİPNOGRAM OLUŞTURMA")
    print(f"{'='*70}")
    print(f"Çıktı klasörü: {output_folder}")
    print(f"Veritabanı: {db_name}")
    print(f"Cache kullanımı: {'Aktif ✓' if use_cache else 'Pasif'}")
    print(f"{'='*70}\n")
    
    # Global cache'i import et
    if use_cache:
        # Hasta listesini belirle
        if patient_ids is None:
            # Cache'de varsa oradan al
            if _STAGES_CACHE:
                patient_ids = sorted(_STAGES_CACHE.keys())
                print(f"Hasta listesi cache'den alındı: {len(patient_ids)} hasta\n")
            else:
                # Cache boşsa MongoDB'den hasta listesini al
                client = MongoClient(mongo_uri)
                db = client[db_name]
                fs = gridfs.GridFS(db)
                
                patient_ids = set()
                for file in fs.find({"metadata.data_type": "sleep_stages"}):
                    patient_ids.add(file.metadata['patient_id'])
                patient_ids = sorted(patient_ids)
                
                client.close()
                
                print(f"Toplam {len(patient_ids)} hasta bulundu (MongoDB'den)\n")
                
                # Tüm stage verilerini önceden yükle
                print("Performans için tüm stage verileri cache'e yükleniyor...")
                preload_all_stages(patient_ids, mongo_uri, db_name)
        else:
            # Belirtilen hastalar için cache'i kontrol et
            missing_patients = [pid for pid in patient_ids if pid not in _STAGES_CACHE]
            
            if missing_patients:
                print(f"Cache'de olmayan {len(missing_patients)} hasta için veri yükleniyor...")
                preload_all_stages(missing_patients, mongo_uri, db_name)
        
        # Cache referansını kullan
        cache_ref = _STAGES_CACHE
    else:
        # Cache kullanılmayacak
        cache_ref = None
        
        # Hasta listesini belirle
        if patient_ids is None:
            client = MongoClient(mongo_uri)
            db = client[db_name]
            fs = gridfs.GridFS(db)
            
            patient_ids = set()
            for file in fs.find({"metadata.data_type": "sleep_stages"}):
                patient_ids.add(file.metadata['patient_id'])
            patient_ids = sorted(patient_ids)
            
            client.close()
            print(f"Toplam {len(patient_ids)} hasta bulundu.\n")
    
    results = {
        'total': len(patient_ids),
        'success': 0,
        'failed': 0,
        'details': []
    }
    
    # Her hasta için hipnogram oluştur
    for idx, patient_id in enumerate(patient_ids, 1):
        print(f"[{idx}/{len(patient_ids)}] İşleniyor: {patient_id}")
        
        try:
            output_path = create_hypnogram(
                patient_id=patient_id,
                mongo_uri=mongo_uri,
                db_name=db_name,
                output_folder=output_folder,
                figsize=figsize,
                dpi=dpi,
                _stages_cache=cache_ref
            )
            
            if output_path:
                results['success'] += 1
                results['details'].append({
                    'patient_id': patient_id,
                    'status': 'success',
                    'output_path': output_path
                })
            else:
                results['failed'] += 1
                results['details'].append({
                    'patient_id': patient_id,
                    'status': 'failed',
                    'reason': 'unknown'
                })
                
        except Exception as e:
            print(f"✗ Hata: {patient_id} - {str(e)}\n")
            results['failed'] += 1
            results['details'].append({
                'patient_id': patient_id,
                'status': 'failed',
                'reason': str(e)
            })
    
    # Özet rapor
    print(f"\n{'='*70}")
    print(f"İŞLEM ÖZETİ")
    print(f"{'='*70}")
    print(f"Toplam hasta: {results['total']}")
    print(f"Başarılı: {results['success']} ✓")
    print(f"Başarısız: {results['failed']} ✗")
    print(f"Çıktı klasörü: {os.path.abspath(output_folder)}")
    
    if use_cache:
        print(f"Cache kullanımı: {len(_STAGES_CACHE)} hasta bellekte tutuldu")
    
    print(f"{'='*70}\n")
    
    if results['failed'] > 0:
        print("Başarısız işlemler:")
        for detail in results['details']:
            if detail['status'] == 'failed':
                print(f"  - {detail['patient_id']}: {detail.get('reason', 'unknown')}")
    
    return results


if __name__ == "__main__":
    # Örnek 1: Tek bir hasta için hipnogram oluştur
    """
    create_hypnogram(
        patient_id="patient_001",
        output_folder="hypnograms"
    )
    """
    
    # Örnek 2: Tüm hastalar için hipnogram oluştur
    
    results = create_all_hypnograms(
        output_folder="hypnograms",
        figsize=(16, 6),
        dpi=150
    )
    
    # Örnek 3: Belirli hastalar için oluştur
    """
    patient_list = ["patient_001", "patient_002", "patient_003"]
    results = create_all_hypnograms(
        patient_ids=patient_list,
        output_folder="my_hypnograms"
    )
    """
    print("Just Now Any Info did not write")
    pass



TOPLU HİPNOGRAM OLUŞTURMA
Çıktı klasörü: hypnograms
Veritabanı: psg_data
Cache kullanımı: Aktif ✓

Hasta listesi cache'den alındı: 17 hasta

[1/17] İşleniyor: 22cd54f1-5d96-4b12-a392-d5ddada9d8d3

HİPNOGRAM OLUŞTURULUYOR: 22cd54f1-5d96-4b12-a392-d5ddada9d8d3
✓ 866 epoch yüklendi (cache'den - MongoDB'ye gidilmedi)
✓ Hipnogram kaydedildi: hypnograms/22cd54f1-5d96-4b12-a392-d5ddada9d8d3_hypnogram.png
  Boyut: 16x6 inch, 150 DPI

[2/17] İşleniyor: 28ba35f0-a068-4eeb-a51a-f4c45b4c0843

HİPNOGRAM OLUŞTURULUYOR: 28ba35f0-a068-4eeb-a51a-f4c45b4c0843
✓ 728 epoch yüklendi (cache'den - MongoDB'ye gidilmedi)
✓ Hipnogram kaydedildi: hypnograms/28ba35f0-a068-4eeb-a51a-f4c45b4c0843_hypnogram.png
  Boyut: 16x6 inch, 150 DPI

[3/17] İşleniyor: 2d42885b-ba49-4c21-9e8f-4963349f252c

HİPNOGRAM OLUŞTURULUYOR: 2d42885b-ba49-4c21-9e8f-4963349f252c
✓ 740 epoch yüklendi (cache'den - MongoDB'ye gidilmedi)
✓ Hipnogram kaydedildi: hypnograms/2d42885b-ba49-4c21-9e8f-4963349f252c_hypnogram.png
  Boyut: 16x6 inch, 

In [3]:
# ====================================================================
# VERİ LİSTELEME/OKUMA/SİLME
# ====================================================================

# Tüm hastaları listele
list_all_patients(
    mongo_uri="mongodb://localhost:27017/",
    db_name="psg_data",
    show_channels_info=True
)



Veritabanında 17 hasta bulundu:

Kanallar:
  - C3 (256.0 Hz, 50.74 MB)
  - C4 (256.0 Hz, 50.74 MB)
  - F3 (256.0 Hz, 50.74 MB)
  - F4 (256.0 Hz, 50.74 MB)
  - O1 (256.0 Hz, 50.74 MB)
  - O2 (256.0 Hz, 50.74 MB)
  - M1 (256.0 Hz, 50.74 MB)
  - M2 (256.0 Hz, 50.74 MB)
  - E1 (256.0 Hz, 50.74 MB)
  - E2 (256.0 Hz, 50.74 MB)
  - CHIN1 (256.0 Hz, 50.74 MB)
  - CHIN2 (256.0 Hz, 50.74 MB)
  - CHINz (256.0 Hz, 50.74 MB)
  - EKG+ (256.0 Hz, 50.74 MB)
  - EKG- (256.0 Hz, 50.74 MB)
  - RLEG+ (256.0 Hz, 50.74 MB)
  - LLEG+ (256.0 Hz, 50.74 MB)
  - Snore (256.0 Hz, 50.74 MB)
  - Thermistor (32.0 Hz, 6.34 MB)
  - Nasal Pressure (256.0 Hz, 50.74 MB)
  - Flow_DR (256.0 Hz, 50.74 MB)
  - Snore_DR (256.0 Hz, 50.74 MB)
  - Abdomen (64.0 Hz, 12.69 MB)
  - Chest (64.0 Hz, 12.69 MB)
  - Phase (64.0 Hz, 12.69 MB)
  - RMI (64.0 Hz, 12.69 MB)
  - RR (64.0 Hz, 12.69 MB)
  - XSum (64.0 Hz, 12.69 MB)
  - XFlow (64.0 Hz, 12.69 MB)
  - XVolume (64.0 Hz, 12.69 MB)
  - Position (4.0 Hz, 0.79 MB)
  - Elevation (4.0 H

In [ ]:
""" delete_patient_data(
    patient_id="53ab11aa-3f94-41c1-9d1e-48767cce3c26",
    mongo_uri="mongodb://localhost:27017/",
    db_name="psg_data"
) """

""" delete_patient_datas(
    patient_ids=["6cca9ca1-7cdd-45ba-9443-6f5dbf921be2",
                 "5849e85b-72f0-4b52-a624-b3cbeffc07d4",
                 "a7e93e13-1253-4134-bd18-977fb61e95af",
                 "bcf4b0c2-5c27-447a-8123-6e3df0ce3297",
                 "eb72b001-3e2f-4ac9-8a4d-a0c108d450b9",
                 "ec59558e-c2d3-445e-8e51-0f9c09ade638",
                 "f9f4b869-e74b-443d-8c87-4ea91c901166"
                 ],
    mongo_uri="mongodb://localhost:27017/",
    db_name="psg_data",
    confirm=False
) """

In [5]:
# Belirli bir hastanın verilerini oku, dosyaya yaz
patient_data = read_patient_data(
    patient_id="22cd54f1-5d96-4b12-a392-d5ddada9d8d3",
    mongo_uri="mongodb://localhost:27017/",
    db_name="psg_data",
    channel_name="Chest"
)

np.savetxt("22cd54f1-5d96-4b12-a392-d5ddada9d8d3_Chest.txt", patient_data["signal_data"]) # Tüm veriyi bir dosyaya yazdırmak için


Hasta ID: 22cd54f1-5d96-4b12-a392-d5ddada9d8d3
Uyku evreleri: 866 epoch
Toplam kanal sayısı: 45

Mevcut kanallar:
  - C3 (256.0 Hz)
  - C4 (256.0 Hz)
  - F3 (256.0 Hz)
  - F4 (256.0 Hz)
  - O1 (256.0 Hz)
  - O2 (256.0 Hz)
  - M1 (256.0 Hz)
  - M2 (256.0 Hz)
  - E1 (256.0 Hz)
  - E2 (256.0 Hz)
  - CHIN1 (256.0 Hz)
  - CHIN2 (256.0 Hz)
  - CHINz (256.0 Hz)
  - EKG+ (256.0 Hz)
  - EKG- (256.0 Hz)
  - RLEG+ (256.0 Hz)
  - LLEG+ (256.0 Hz)
  - Snore (256.0 Hz)
  - Thermistor (32.0 Hz)
  - Nasal Pressure (256.0 Hz)
  - Flow_DR (256.0 Hz)
  - Snore_DR (256.0 Hz)
  - Abdomen (64.0 Hz)
  - Chest (64.0 Hz)
  - Phase (64.0 Hz)
  - RMI (64.0 Hz)
  - RR (64.0 Hz)
  - XSum (64.0 Hz)
  - XFlow (64.0 Hz)
  - XVolume (64.0 Hz)
  - Position (4.0 Hz)
  - Elevation (4.0 Hz)
  - Activity (4.0 Hz)
  - PPG (256.0 Hz)
  - Plethysmogram (128.0 Hz)
  - Flow (256.0 Hz)
  - EPAP (256.0 Hz)
  - IPAP (256.0 Hz)
  - SpO2 (128.0 Hz)
  - Pulse (128.0 Hz)
  - RD-Quality (128.0 Hz)
  - Gravity X (16.0 Hz)
  - Gravity Y

# KANAL İŞLEMLERİ

In [ ]:

# ===================================================================
# ÖRNEK: Tek kanal, tek epoch aralığı - RAW DATA
# ===================================================================
print("\n\n")
print("=" * 80)
print("ÖRNEK 5: Tek kanal, tek epoch aralığı - RAW DATA")
print("=" * 80)

""" result_whole_data = extract_epoch_range_whole_data_for_validation(
    patient_id="3da2681a-5991-449c-80a4-52cf3257db02",
    channel_name="Activity",
    start_epoch=17,
    end_epoch=23,
    save_to_file=True,
    output_format='txt'
) """

""" result_with_partial_read = extract_epoch_range_data_v2(
    patient_id="3da2681a-5991-449c-80a4-52cf3257db02",
    channel_name="Activity",
    start_epoch=17,
    end_epoch=23,
    save_to_file=True,
    output_format='txt'
) """

""" if result_with_partial_read:
    print(f"\nVeri çekildi:")
    print(f"  Signal shape: {result_with_partial_read['signal_data'].shape}")
    print(f"  Sample frequency: {result_with_partial_read['sample_frequency']} Hz")
    print(f"  Duration: {result_with_partial_read['duration']:.2f} seconds") """



In [ ]:
""" # Örnek: Aynı hasta için tekrar çekme (cache'den alır - çok hızlı)
result1 = extract_epoch_range_data_v2(
    patient_id='patient_001',  # Aynı hasta
    channel_name='01',       # Farklı kanal
    start_epoch=200,
    end_epoch=249,
    mongo_uri="mongodb://localhost:27017/",
    db_name="psg_data",
    output_format="txt"
) """
    
    